In [ ]:
# The pip install can take a minute
%pip install -q urllib3<2.0 datascience ipywidgets
import pyodide_http
pyodide_http.patch_all()

from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

# sec 1 - Tables

Content covered:
1. creating tables 
2. selecting 
3. filtering 
4. sorting 
5. chaining

Covered by videos 5-7. 
Assumes you have gone through `sec01_python` and `sec01_arrays`.

---

## 1. Two ways to create a table

**Way 1:** Reading from a file

**Syntax - reading a table from a file**

```
Table.read_table('filename.csv')
table.show(n)
table.show()
```

- The filename is a **string**, so it goes in quotes.
- The file must be in the same folder as the notebook.
- `.show(n)` displays the first *n* rows; `.show()` displays all of them.
- Always look at a few rows before doing anything else to a table you have not seen.

In [ ]:
cones = Table.read_table('cones.csv')
cones

In [ ]:
cones.show(3)

In [ ]:
cones.show()

**Way 2:** Creating from scratch

**Syntax - building a table from scratch**

```
Table()
table.with_column(label, values)
table.with_columns(label1, values1, label2, values2, ...)
```

- `Table()` creates an empty table.
- `label` is a **string**; `values` is an **array**.
- Every column must have the **same length**.
- These return a **new** table. The original is unchanged unless you assign the result.

In [ ]:
streets = make_array('Bancroft', 'Durant', 'Channing', 'Haste')
streets

In [ ]:
Table()

In [ ]:
southside = Table().with_column('Streets', streets)
southside

In [ ]:
southside.with_column('Blocks from campus', np.arange(4))

In [ ]:
Table().with_columns(
    'Streets', streets,
    'Blocks from campus', np.arange(4)
)

## 2. Table operations

**Syntax - choosing columns**

| Form | Keeps | Returns |
|---|---|---|
| `table.select(label, ...)` | only the columns named | a table |
| `table.drop(label, ...)` | everything **except** those named | a table |

- Column labels are **strings**. They always go in quotes.
- `table.select(Flavor)` without quotes is a `NameError` - Python looks for a name,
  not a label.
- Both return a **new** table. The original keeps all its columns.

In [ ]:
cones.select('Flavor')

In [ ]:
cones.select('Flavor', 'Price')

The line below causes an error because `Flavor` is not a name, but rather, a *column* in the `cones` table.

In [ ]:
cones.select(Flavor, 'Price')

In [ ]:
cones.drop('Price')

In [ ]:
cones

In [ ]:
cones_without_price = cones.drop('Price')
cones_without_price

**Syntax - choosing rows**

```
table.where(column_label, value)
```

- `select` and `drop` work on **columns**. `where` works on **rows**.
- Both arguments are usually strings, and both go in quotes.
- Every column comes back - only the matching rows are kept.
- If nothing matches you get an **empty table**, not an error. Check the column's
  actual values before filtering.

In [ ]:
cones.where('Flavor', 'chocolate')

**Syntax - ordering rows**

```
table.sort(column_label)
table.sort(column_label, descending=True)
```

- Ascending by default. `descending=True` is a named argument, like `ndigits` on `round`.
- Works on numbers and on text (alphabetical).
- Returns a **new** table.

In [ ]:
cones.sort('Price')

In [ ]:
cones.sort('Price', descending=True)

In [ ]:
cones.sort('Flavor', descending=True)

## 3. A real data set: the 2026 World Cup

In [ ]:
# From https://www.kaggle.com/datasets/troubador/2026-world-cup-player-statistics
# One row per player at the 2026 World Cup.
wc = Table.read_table('wcplayerstatistics2026.csv')
wc

In [ ]:
wc.show(6)

In [ ]:
# Who scored the most goals at the tournament?
wc.sort('Gls', descending=True).show(6)

### Filtering and sorting

In [ ]:
# Pos holds clean two-letter codes: GK, DF, MF, FW
forwards = wc.where('Pos', 'FW')

In [ ]:
forwards

In [ ]:
forwards.drop('Pos')

In [ ]:
forwards

In [ ]:
# Drop the bookkeeping columns we will not use
forwards = forwards.drop('Rk', 'Pos', 'PlayerID', 'SecondPos')

In [ ]:
forwards.show(10)

In [ ]:
forwards.sort('Min').show(10)

In [ ]:
forwards.sort('Min', descending=True).show(10)

### Your turn

**Task:** Extract the rows in the `wc` table for Egypt's squad. Save the table into the name `egypt`.

<details>
<summary><strong>Answer</strong></summary>

<code>egypt = wc.where('Squad', 'eg Egypt')</code>

If you tried <code>wc.where('Squad', 'Egypt')</code> you got an empty table with no
error message. The stored values carry a two-letter country-code prefix. The cells
below work through why, and what habit prevents it.

</details>


<details>
<summary><strong>Stuck on the task above? Open this.</strong></summary>

**Careful - look at the actual values before you filter.** The `Squad` column stores a country code in front of the country name.

</details>


**Syntax - reading one column as an array**

```
table.column(column_label)
table.column(column_label).item(index)
```

- Returns the column's values as an **array**, not as a table.
- Chain `.item(0)` on the end to inspect a single value.
- Use this to check what a column *actually contains* before you filter on it.

In [ ]:
# Squad is not just the country name
wc.column('Squad').item(0)

In [ ]:
# This returns an EMPTY table - 'Egypt' is not the stored value
wc.where('Squad', 'Egypt')

In [ ]:
# The stored value includes the 'eg ' prefix
egypt = wc.where('Squad', 'eg Egypt')
egypt.show(5)

**Discussion Question**: `Club` is stored the same way (e.g. `1.eng Leeds United`). What would `wc.where('Club', 'Leeds United')` return, and why?

*Write your prediction here before running the cell below, then check yourself:*


<details>
<summary><strong>Answer</strong></summary>

An <strong>empty table</strong>, with no error message.

<code>Club</code> values carry a league prefix ; <code>1.eng Leeds United</code> ;
so the bare name matches nothing. This case is harder than <code>Squad</code> because the
prefix format varies by league and division, so it cannot be guessed. Inspect the column.

</details>


In [ ]:
# Chaining two filters: Egypt's forwards only
wc.where('Squad', 'eg Egypt').where('Pos', 'FW')

## 4. `.select()` returns a table - `.column()` returns an array

**Syntax - table out versus array out**

| Form | Returns | Use it when |
|---|---|---|
| `table.select(label)` | a **table** with one column | you will keep working with it as a table |
| `table.column(label)` | an **array** of values | you will do arithmetic on it |

- Same data, two containers. Tell them apart by how they display: a table has a
  heading and a border, an array has square brackets and commas.
- Array functions such as `np.average` need `.column`, not `.select`.

In [ ]:
wc.select('Min')

In [ ]:
type(wc.select('Min'))

In [ ]:
wc.column('Min')

In [ ]:
type(wc.column('Min'))

##### Remember that a column of a table is an array!

In [ ]:
# Remember: .column() gives an array, so np.average works on it
np.average(wc.column('Min'))

**Discussion Question:**
- Which of the following two lines of code correctly calculates the average minutes played by Egypt's squad?
- Use the [Python reference sheet](https://www.data8.org/sp25/reference/) to help you answer!

*Write your prediction here before running the cell below, then check yourself:*


<details>
<summary><strong>Answer</strong></summary>

<code>np.average(egypt.column('Min'))</code>.

<code>.select</code> returns a <strong>Table</strong>, which <code>np.average</code> cannot
work with. <code>.column</code> returns an <strong>array</strong>, which is what array
functions expect.

The rule: if you are going to do arithmetic on it, you want <code>.column</code>.

</details>


In [ ]:
np.average(egypt.select('Min'))

In [ ]:
np.average(egypt.column('Min'))

**Task:** Who played the most minutes for Egypt? Return just the player's name, not the whole row.

<details>
<summary><strong>Answer</strong></summary>

<code>egypt.sort('Min', descending=True).column('Player').item(0)</code>

Sort, take the column as an array, take the first element. Note <code>.item(0)</code>
and not <code>.item(1)</code> ; arrays count from zero.

</details>


**Discussion Question:** `egypt.sort('Min', descending=True).column('Player')` and `egypt.sort('Min', descending=True).select('Player')` both run without error. What is different about what they give you back?

*Write your prediction here before running the cell below, then check yourself:*


<details>
<summary><strong>Answer</strong></summary>

Both run without error. The difference is what you get back.

<code>.column('Player')</code> gives an <strong>array</strong> of names, which you can index
with <code>.item(0)</code>. <code>.select('Player')</code> gives a <strong>one-column
table</strong>, which you cannot.

This is the pattern with these two: the mistake is silent where you make it and
surfaces one step later.

</details>


## 5. The order of operations matters

**Syntax - method chaining**

```
table.method1(...).method2(...)
```

- Read **left to right**. Each method runs on whatever the previous one produced.
- At every step ask: what kind of thing do I have now, and does it still have what
  the next step needs?
- Order matters. Dropping a column before filtering on it is an error.

**Discussion Question**: Which one of the following two lines of code fail?

*Write your prediction here before running the cell below, then check yourself:*


<details>
<summary><strong>Answer</strong></summary>

The <strong>first</strong> line fails:
<code>wc.drop('Pos').where('Pos', 'FW')</code>.

Chains read <strong>left to right</strong>. The drop removes the column before the filter
looks for it. Filter first, drop afterwards.

</details>


In [ ]:
wc.drop('Pos').where('Pos', 'FW')

In [ ]:
wc.where('Pos', 'FW').drop('Pos')

## 6. Real data brings type surprises

#### The file stores whole numbers as `floats`

Ages, birth years and match counts all arrive with a `.0` on the end. They are `float`, not `int`.

In [ ]:
# Age reads as 25.0, not 25
wc.column('Age').item(0)

In [ ]:
# Confirm the type
type(wc.column('Age').item(0))

**Review Question**: `Born` holds values like `2000.0`. Which of the next two lines gives you the integer year, and which one errors?

*Write your prediction here before running the cell below, then check yourself:*


<details>
<summary><strong>Answer</strong></summary>

<strong>Line A</strong> works and gives <code>2000</code>.
<code>int</code> on a float truncates the decimal part, and here it is zero.

<strong>Line B</strong> fails. <code>str</code> produces the string <code>'2000.0'</code>,
and <code>int</code> cannot parse a string containing a decimal point ; ValueError.

For a numeric string, go via float: <code>int(float(...))</code>.

</details>


In [ ]:
# Line A
int(wc.column('Born').item(0))

In [ ]:
# Line B
int(str(wc.column('Born').item(0)))

## 7. W.E.B. DuBois was a data scientist!

In [ ]:
du_bois = Table.read_table('du_bois.csv')
du_bois

**Task**: Find the number of rows and the number of columns in this table.

<details>
<summary><strong>Answer</strong></summary>

<code>du_bois.num_rows</code> and <code>du_bois.num_columns</code>.

Note there are no brackets. These are <em>attributes</em>, not methods ; you are
asking the table for a property it already knows, not asking it to compute something.

</details>


**Task:** Find all of the variables recorded on each "individual" in this table.

<details>
<summary><strong>Answer</strong></summary>

<code>du_bois.labels</code>

Returns the column labels. Also an attribute, so again no brackets.

</details>


**Task:** Extract just the information (row(s)) of the families with the highest earning income bracket on average, circa 1900. 
- There are multiple correct ways to do this! See if you can find more than one.

<details>
<summary><strong>Answer</strong></summary>

Two routes, both correct.

<strong>Way 1 ; sort.</strong> Sort by the income column in descending order and take
the top row with <code>.take(0)</code>.

<strong>Way 2 ; filter.</strong> Use <code>.where</code> on the class column if you
already know the label of the highest bracket.

Sorting is safer here because it does not require you to know the label in advance.
Run <code>du_bois.labels</code> first if you are unsure what the columns are called.

</details>


Way 1:

Way 2:

Discussion [1 min]

**Challenge Task pt 1:** Based on the relevant variables in the table, estimate the amount of money each that each income class of Black Americans spent on food, circa 1900.

**Challenge Task pt 2:** Add these estimates to the `du_bois` table as a new column.

---